In [24]:
#import os
import pandas as pd
import numpy as np
import pypandoc


import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [25]:
from sklearn.feature_extraction.text import CountVectorizer
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Claudia\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [26]:
import docx
from docx import Document
import os

In [27]:
directory_path = os.getcwd()  # Get the current directory path
directory_path

'c:\\Users\\Claudia\\OneDrive - University of Connecticut\\Documents\\Uconn_Class\\strategic_planning'

In [41]:
#remove failed_parse = 1
#we want 0
documents_df = pd.read_csv('documents_df.csv', delimiter='|')
documents_df.head()

,district,pages,ocr,text,filename,contains_alphanumeric,failed_parse
0,Smethport Area SD,9,0,Smethport Area SD District Level Plan 07/01/20...,Smethport Area SD.csv,True,0
1,Bibb County,22,0,#Built4Bibb: More Victory Planned 2023-2028 St...,Bibb County.csv,True,0
2,Fairview SD 72,1,0,Community Connections and Relations - Develop ...,Fairview SD 72.csv,True,0
3,SLATON ISD,40,0,Slaton Independent School District District Im...,SLATON ISD.csv,True,0
4,Springfield,45,0,Reimagining School to Realize the Portrait of ...,Springfield.csv,True,0


In [29]:
parameters_df = pd.read_csv('hyperparameters_models.csv', delimiter=',')
parameters_df.head()

,Unnamed: 0,chunk,stop,diy_gram,tribigram,min_df,lemma,tfidf,topics
0,0,0,0,0,0,1,0,0,10
1,1,0,0,0,0,1,0,0,80
2,2,0,0,0,0,1,0,1,10
3,3,0,0,0,0,1,0,1,80
4,4,0,0,0,0,1,1,0,10


In [30]:
parameters_df.chunk.value_counts()

chunk
0      192
100    192
150    192
200    192
Name: count, dtype: int64

notes
* remove stop words
* diy_gram julia words
* stop 1 or 0 whatever or not to remove stopwords
* min_df

In [31]:
parameters_test = parameters_df.sample(50,random_state= 10)
parameters_test

,Unnamed: 0,chunk,stop,diy_gram,tribigram,min_df,lemma,tfidf,topics
568,568,150,1,1,1,20,0,0,10
620,620,200,0,0,1,20,1,0,10
456,456,150,0,1,1,1,0,0,10
197,197,100,0,0,0,1,1,0,80
714,714,200,1,0,1,20,0,1,10
27,27,0,0,0,1,1,0,1,80
277,277,100,0,1,1,5,1,0,80
64,64,0,0,1,0,20,0,0,10
720,720,200,1,1,0,1,0,0,10
475,475,150,0,1,1,20,0,1,80


In [32]:
len(documents_df)

615

#Create a code to chunk 100, 150, 200 create a code

##Doc 100


In [35]:
if 'text_lower' not in documents_df.columns:
    documents_df['text_lower'] = documents_df['text'].str.lower()

In [42]:
documents_df['text_lower'] = documents_df['text'].str.lower()

In [50]:
documents_df.head()

,district,pages,ocr,text,filename,contains_alphanumeric,failed_parse,text_lower
0,Smethport Area SD,9,0,Smethport Area SD District Level Plan 07/01/20...,Smethport Area SD.csv,True,0,smethport area sd district level plan 07/01/20...
1,Bibb County,22,0,#Built4Bibb: More Victory Planned 2023-2028 St...,Bibb County.csv,True,0,#built4bibb: more victory planned 2023-2028 st...
2,Fairview SD 72,1,0,Community Connections and Relations - Develop ...,Fairview SD 72.csv,True,0,community connections and relations - develop ...
3,SLATON ISD,40,0,Slaton Independent School District District Im...,SLATON ISD.csv,True,0,slaton independent school district district im...
4,Springfield,45,0,Reimagining School to Realize the Portrait of ...,Springfield.csv,True,0,reimagining school to realize the portrait of ...


In [51]:
def split_text_into_chunks(text):
    words = text.split()
    chunks_split = [words[i:i + 100] for i in range(0, len(words), 150)]
    return [' '.join(chunk_split) for chunk_split in chunks_split]

In [53]:
# Create new documents
new_documents = []

for index, row in documents_df.iterrows():
    text_lower = row['text_lower']
    chunks = split_text_into_chunks(text_lower)
    
    for i, chunk in enumerate(chunks):
        new_document = {
            'district': row['district'],
            'chunk_index': i + 1,
            'chunk_text': chunk
        }
        new_documents.append(new_document)

In [54]:
new_documents[1:5]

[{'district': 'Smethport Area SD',
  'chunk_index': 2,
  'chunk_text': 'keystone advanced rates at or above the pa state average in math, ela and science. type: annual data source: future ready index, pvaas specific targets: pvaas growth data showing evidence that the district has met the standard for growth in each year (green) and for the 3 year average in pssa grade 4 math, pssa grade 5 math, pssa grade 6 math and pssa grade 7 math. 110 pvaas growth data showing evidence that the district has met the standard for growth in each year (green) and for the 3 year average in pssa grade 5 ela, pssa grade 7'},
 {'district': 'Smethport Area SD',
  'chunk_index': 3,
  'chunk_text': 'strategies: common assessment within grade/subject description: wwc reports the effective use of data can have a positive impact upon student achievement; using common assessments to inform teacher practice is one such use of data. (source: http://ies.ed.gov/ncee/wwc/pdf/practice_guides/dddm_pg_092909.pdf?) teach

In [55]:
new_df = pd.DataFrame(new_documents)
new_df 

,district,chunk_index,chunk_text
0,Smethport Area SD,1,smethport area sd district level plan 07/01/20...
1,Smethport Area SD,2,keystone advanced rates at or above the pa sta...
2,Smethport Area SD,3,strategies: common assessment within grade/sub...
3,Smethport Area SD,4,of testimonials and classroom examples of posi...
4,Smethport Area SD,5,safe and supportive schools implementation ste...
...,...,...,...
12653,Davidson County,9,strengthen the core curriculum at all levels s...
12654,Davidson County,10,opportunities for advanced level courses and t...
12655,Davidson County,11,into instructional design metric 1: visual rep...
12656,Davidson County,12,"sources, was compiled. the initial work of the..."


In [59]:
#strategic plans 
chunk_counts = new_df.groupby('district')['chunk_index'].count().reset_index()

# Rename the count column to 'total_chunks' for clarity
chunk_counts = chunk_counts.rename(columns={'chunk_index': 'total_chunks'})

# Display the result
print(chunk_counts)

                       district  total_chunks
0                   ABILENE ISD             4
1    ALBUQUERQUE PUBLIC SCHOOLS            29
2                      ANADARKO            55
3                  ANGLETON ISD             7
4                ARAPAHO-BUTLER             4
..                          ...           ...
567                   Worcester            28
568         YORK PUBLIC SCHOOLS            37
569                     York 01             6
570                York City SD             5
571             Youngstown City            71

[572 rows x 2 columns]


In [57]:
new_df.value_counts()
len(new_df)

12658

In [60]:
from sklearn.feature_extraction.text import CountVectorizer

#CountVect


In [62]:
#only split based on " "
#def split_string(text):
    #tokens = text.split(" ")  
    #return tokens

In [76]:
import re

def split_string(text):
    # Remove all punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)  # Keep only alphanumeric and spaces
    # Tokenize by splitting on whitespace
    return text.split()

In [77]:
new_df["tokens_text"]= new_df['chunk_text'].apply(split_string)


In [78]:
new_df

,district,chunk_index,chunk_text,tokens_text
0,Smethport Area SD,1,smethport area sd district level plan 07/01/20...,"[smethport, area, sd, district, level, plan, 0..."
1,Smethport Area SD,2,keystone advanced rates at or above the pa sta...,"[keystone, advanced, rates, at, or, above, the..."
2,Smethport Area SD,3,strategies: common assessment within grade/sub...,"[strategies, common, assessment, within, grade..."
3,Smethport Area SD,4,of testimonials and classroom examples of posi...,"[of, testimonials, and, classroom, examples, o..."
4,Smethport Area SD,5,safe and supportive schools implementation ste...,"[safe, and, supportive, schools, implementatio..."
...,...,...,...,...
12653,Davidson County,9,strengthen the core curriculum at all levels s...,"[strengthen, the, core, curriculum, at, all, l..."
12654,Davidson County,10,opportunities for advanced level courses and t...,"[opportunities, for, advanced, level, courses,..."
12655,Davidson County,11,into instructional design metric 1: visual rep...,"[into, instructional, design, metric, 1, visua..."
12656,Davidson County,12,"sources, was compiled. the initial work of the...","[sources, was, compiled, the, initial, work, o..."


In [74]:
vectorizer = CountVectorizer(tokenizer=split_string, lowercase=True)
X = vectorizer.fit_transform(new_df["chunk_text"])

c:\Users\Claudia\anaconda3\lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [75]:
print(vectorizer.get_feature_names_out())

['0' '00' '000' ... 'zweaverbrockwayk12paus'
 'zyicrtu_uystr_zuhrxtubzucvtxsciyzucsuyscvvjarajr__gxtyiz_rvzk_tul'
 'zytniowski']


In [70]:
print(X.toarray())

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
